# Verify boundary-overlap token selection

Visual + quantitative sanity check for `embeddings_cell` / `embeddings_nucleus`
(see `InferenceProvider.pool_boundary_tokens` in
`src/python/extract_embeddings/inference_providers/inference_provider.py`).

For a handful of cells, this draws the raw centre patch with:
- the model's token grid,
- the Xenium **cell** boundary polygon (blue) and **nucleus** boundary polygon (orange),
  both mapped into H&E pixel space,
- the tokens whose footprint overlaps each polygon (filled, matching color),
- the fixed **four central tokens** (`patches_to_save`, black dashed) for comparison.

If a polygon has no overlapping token, the nearest-to-centroid fallback token is
drawn hatched, and a warning is printed — matching the `WARNING: no {kind} token
overlap ... falling back to nearest token` printed during real extraction.

A later cell scans a larger sample of cells and reports how often each boundary
type falls back, as a coarse check that the alignment matrix / coordinate
convention is correct (a high fallback rate usually means something's off).


In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT))


def _load_dotenv(path: Path) -> None:
    """Minimal `.env` loader (KEY=value per line) -- avoids an extra dependency.
    Mirrors how every other entry point in this repo expects env vars to be set
    (see src/python/code_configs/paths.py)."""
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key, value = key.strip(), value.strip()
        if value:
            os.environ.setdefault(key, value)


_load_dotenv(REPO_ROOT / ".env")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon as MplPolygon

from src.python.code_configs import paths
from src.python.extract_embeddings.data.patch_dataset import PatchDataset
from src.python.extract_embeddings.data.xenium_boundaries import token_overlap_mask, nearest_token


## Config

Pick a **raw** (non-`converted`) dataset -- boundary-polygon mapping goes through
the Xenium<->H&E alignment matrix the same way `ResizedCellDataset` uses it, which
today is only exercised for raw WSIs (see `extract_embeddings.py`'s entry point).
`MODEL_GRID` mirrors each provider's `(token_size, grid_size, patches_to_save)`
exactly as hardcoded in its `__init__` -- keep these in sync if you change a
provider's defaults. CellViT's `token_size` comes from its checkpoint
(`self.model.patch_size`) at load time; 16 below is HIPT/SAM's usual value --
override it if your checkpoint differs.


In [ ]:
DATASET_NAME = "Xenium_V1_Human_Colon_Cancer_P1_CRC_Add_on_FFPE"
WSI_FILENAME = f"{DATASET_NAME}_he_image.ome.tif"

MODEL_GRID = {
    "UNI2":       dict(token_size=14, grid_size=16, x_size=224, y_size=224, offset_x=0,    offset_y=0,
                        central_tokens={"top_left": (7, 7), "top_right": (7, 8), "bottom_left": (8, 7), "bottom_right": (8, 8)}),
    "VirchowV2":  dict(token_size=14, grid_size=16, x_size=224, y_size=224, offset_x=0,    offset_y=0,
                        central_tokens={"top_left": (7, 7), "top_right": (7, 8), "bottom_left": (8, 7), "bottom_right": (8, 8)}),
    "HOptimus1":  dict(token_size=14, grid_size=16, x_size=224, y_size=224, offset_x=0,    offset_y=0,
                        central_tokens={"top_left": (7, 7), "top_right": (7, 8), "bottom_left": (8, 7), "bottom_right": (8, 8)}),
    "CellViT":    dict(token_size=16, grid_size=16, x_size=224, y_size=224, offset_x=0,    offset_y=0,
                        central_tokens={"top_left": (7, 7), "top_right": (7, 8), "bottom_left": (8, 7), "bottom_right": (8, 8)}),
    "CONCH":      dict(token_size=16, grid_size=28, x_size=448, y_size=448, offset_x=-224, offset_y=-224,
                        central_tokens={"top_left": (13, 13), "top_right": (13, 14), "bottom_left": (14, 13), "bottom_right": (14, 14)}),
    "CTransPath": dict(token_size=32, grid_size=7,  x_size=224, y_size=224, offset_x=0,    offset_y=0,
                        central_tokens={"center": (3, 3)}),
}

MODEL = "UNI2"
grid_cfg = MODEL_GRID[MODEL]

cells_info_path = os.path.join(paths.XENIUM_PROCESSED_OUTPUT_ROOT, DATASET_NAME, "patch_coordinates.h5")
cells_csv_path = os.path.join(paths.XENIUM_OUTPUT_ROOT, f"{DATASET_NAME}_out", "cell_boundaries.csv.gz")
nucleus_boundaries_path = os.path.join(paths.XENIUM_OUTPUT_ROOT, f"{DATASET_NAME}_out", "nucleus_boundaries.parquet")
alignment_matrix_path = os.path.join(paths.ALIGNMENT_MATRIX_ROOT, f"{DATASET_NAME}_he_imagealignment.csv")
wsi_path = os.path.join(paths.WSI_RAW_ROOT, WSI_FILENAME)


In [ ]:
dataset = PatchDataset(
    wsi_path=wsi_path,
    cells_info_path=cells_info_path,
    x_size=grid_cfg["x_size"],
    y_size=grid_cfg["y_size"],
    offset_x=grid_cfg["offset_x"],
    offset_y=grid_cfg["offset_y"],
    cells_csv_path=cells_csv_path,
    nucleus_boundaries_path=nucleus_boundaries_path,
    alignment_matrix_path=alignment_matrix_path,
)
len(dataset)


## Single-cell plot

`plot_sample(idx)` draws one cell: token grid, both boundary polygons (translated
into patch-local pixel coordinates), the tokens each one overlaps, and the fixed
central tokens. A dataset_stats-style print underneath reports which kinds (if
any) had to fall back to the nearest token.


In [ ]:
_KIND_COLORS = {"cell": "tab:blue", "nucleus": "tab:orange"}


def plot_sample(idx: int, model: str = MODEL, ax=None):
    cfg = MODEL_GRID[model]
    token_size, grid_size = cfg["token_size"], cfg["grid_size"]

    raw_patch = dataset.get_raw_patch(idx)
    x0, y0 = dataset.origin(idx)
    cell_id = dataset.cell_ids_dataset[idx]
    cell_id_str = cell_id.decode("utf-8") if isinstance(cell_id, bytes) else str(cell_id)

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(raw_patch)

    for t in range(0, cfg["x_size"] + 1, token_size):
        ax.axvline(t, color="white", linewidth=0.5, alpha=0.5)
        ax.axhline(t, color="white", linewidth=0.5, alpha=0.5)

    for kind, color in _KIND_COLORS.items():
        polygon = dataset.boundary_polygon(idx, kind)
        if polygon is None:
            continue

        local_xy = np.array(polygon.exterior.coords) - [x0, y0]
        ax.add_patch(MplPolygon(local_xy, closed=True, facecolor="none", edgecolor=color, linewidth=2))

        mask = token_overlap_mask(polygon, x0, y0, token_size, grid_size)
        if mask.any():
            for r, c in zip(*np.where(mask)):
                ax.add_patch(Rectangle((c * token_size, r * token_size), token_size, token_size,
                                        facecolor=color, alpha=0.3, edgecolor="none"))
        else:
            r, c = nearest_token(polygon, x0, y0, token_size, grid_size)
            print(f"WARNING: no {kind} token overlap for sample idx={idx} -- "
                  f"falling back to nearest token (row={r}, col={c}).")
            ax.add_patch(Rectangle((c * token_size, r * token_size), token_size, token_size,
                                    facecolor="none", edgecolor=color, hatch="///", linewidth=2))

    for r, c in cfg["central_tokens"].values():
        ax.add_patch(Rectangle((c * token_size, r * token_size), token_size, token_size,
                                facecolor="none", edgecolor="black", linestyle="--", linewidth=1.5))

    ax.set_title(f"idx={idx}  cell_id={cell_id_str}  model={model}")
    ax.set_xlim(0, cfg["x_size"])
    ax.set_ylim(cfg["y_size"], 0)
    ax.set_xticks([]); ax.set_yticks([])
    if own_fig:
        plt.show()


In [ ]:
SAMPLE_INDICES = [0, 1, 2, 3, 4, 5]

fig, axes = plt.subplots(2, 3, figsize=(16, 11))
for idx, ax in zip(SAMPLE_INDICES, axes.flat):
    plot_sample(idx, ax=ax)
plt.tight_layout()
plt.show()


## Fallback-rate scan

Runs the same overlap check (no plotting) over a larger sample of cells and
reports, per boundary kind, how often no token overlapped and the fallback token
had to be used. A high rate is a red flag for a wrong alignment matrix / mpp /
coordinate-frame assumption rather than genuinely tiny nuclei.


In [ ]:
N_CHECK = min(500, len(dataset))
rng = np.random.default_rng(0)
check_indices = rng.choice(len(dataset), size=N_CHECK, replace=False)

cfg = MODEL_GRID[MODEL]
counts = {"cell": {"total": 0, "missing_polygon": 0, "fallback": 0},
          "nucleus": {"total": 0, "missing_polygon": 0, "fallback": 0}}

for idx in check_indices:
    x0, y0 = dataset.origin(int(idx))
    for kind in ("cell", "nucleus"):
        counts[kind]["total"] += 1
        polygon = dataset.boundary_polygon(int(idx), kind)
        if polygon is None:
            counts[kind]["missing_polygon"] += 1
            continue
        mask = token_overlap_mask(polygon, x0, y0, cfg["token_size"], cfg["grid_size"])
        if not mask.any():
            counts[kind]["fallback"] += 1

for kind, c in counts.items():
    print(f"{kind}: {c['total']} checked, "
          f"{c['missing_polygon']} missing polygon ({c['missing_polygon'] / c['total']:.1%}), "
          f"{c['fallback']} fell back to nearest token ({c['fallback'] / c['total']:.1%})")
